Import Library

In [1]:
import os, re
import pandas as pd
import json

Config

In [2]:
TRAIN_CSV      = "c:/Users/HP/Downloads/dataset CPICANN/anno_train.csv"
VAL_CSV        = "c:/Users/HP/Downloads/dataset CPICANN/anno_val.csv"
TRAIN_BACK_DIR = "c:/Users/HP/Downloads/dataset CPICANN/train_with_back/train_with_back"
VAL_BACK_DIR   = "c:/Users/HP/Downloads/dataset CPICANN/val_with_back/val_with_back"

COL_DATA_ID     = "dataId"
COL_FORMULA     = "formula"
COL_CRYSTAL_SYS = "crystalSys"

CRYSTAL_SYS_MAP    = {0: "Kubik", 1: "Tetragonal", 2: "Ortorombik", 5: "Monoklinik"}
TARGET_CRYSTAL_SYS = [0, 1, 2, 5]

TWO_THETA_MIN = 10.0
TWO_THETA_MAX = 80.0
N_POINTS      = 4500

BATCH_SIZE  = 32
EPOCHS      = 110
TEST_SIZE   = 0.20
RANDOM_SEED = 42

PALETTE = ["#2196F3", "#F32121", "#09F50D", "#DE21F3"]

Definisi Elemen

In [3]:
A_SITE = {"Li", "Na", "K", "Rb", "Cs", "Mg", "Ca", "Sr", "Ba", "La", "Ce", "Pr", "Nd", "Sm", "Eu", "Gd", "Tb", "Dy", "Ho", "Er", "Tm", "Yb", "Lu", "Y", "Bi", "Th", "Tl", "Ag", "Cd", "U", "Np", "Pu", "Am", "Cm", "Pa", "Hg"}
B_SITE = {"Sc", "Ti", "V",  "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn", "Nb", "Mo", "Ru", "Rh", "Pd", "Ag", "Hf", "Ta", "W",  "Re", "Os", "Ir", "Pt", "Al", "Ga", "In", "Ge", "Sn", "Pb", "Bi", "Zr", "Si", "Mg", "Cd", "Sc", "Tc", "Au", "Sb", "B"}
X_SITE = {"O", "F", "Cl", "Br", "I", "S", "Se", "Te", "N", "P", "H"}

Fungsi Filter

In [4]:
def parse_formula(formula: str) -> dict:
    s = str(formula)
    for _ in range(20):
        m = re.search(r"\(([^()]+)\)(\d*\.?\d*)", s)
        if not m:
            break
        multiplier = float(m.group(2)) if m.group(2) else 1.0
        inner = ""
        for elem, count in re.findall(r"([A-Z][a-z]?)(\d*\.?\d*)", m.group(1)):
            new_count = (float(count) if count else 1.0) * multiplier
            inner += f"{elem}{new_count:.6g}"
        s = s[:m.start()] + inner + s[m.end():]
    pattern = r"([A-Z][a-z]?)(\d*\.?\d*)"
    elements = {}
    for elem, count in re.findall(pattern, str(formula)):
        count = float(count) if count else 1.0
        elements[elem] = elements.get(elem, 0) + count
    return elements

def is_ABX3(formula: str) -> tuple:
    if not formula or str(formula).strip() in ("", "nan"):
        return False, "Formula kosong"
    elems = parse_formula(formula)
    if not elems:
        return False, "Tidak bisa di-parse"

    unknown = {e for e in elems if e not in A_SITE | B_SITE | X_SITE}
    if unknown:
        return False, f"Elemen tidak dikenali: {unknown}"

    a_elems = {e: n for e, n in elems.items() if e in A_SITE and e not in B_SITE}
    b_elems = {e: n for e, n in elems.items() if e in B_SITE}
    x_elems = {e: n for e, n in elems.items() if e in X_SITE}

    n_a = sum(a_elems.values())
    n_b = sum(b_elems.values())
    n_x = sum(x_elems.values())

    if n_a == 0: return False, "Tidak ada elemen A-site"
    if n_b == 0: return False, "Tidak ada elemen B-site"
    if n_x == 0: return False, "Tidak ada elemen X-site"

    ratio_a = n_a / n_b
    ratio_x = n_x / n_b

    if abs(ratio_a - 1.0) <= 0.15 and abs(ratio_x - 3.0) <= 0.30:
        return True, "ABX3"

    return False, f"Rasio A:B:X = {n_a:.2f}:{n_b:.2f}:{n_x:.2f} (bukan ABX3)"

def filter_perovskite_df(csv_path: str, label: str = "dataset") -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    print(f"\n📂 [{label}] Dimuat: {len(df)} baris, {len(df.columns)} kolom")
    
    results               = df[COL_FORMULA].apply(is_ABX3)
    df["is_perovskite"]   = results.apply(lambda x: x[0])
    df["perovskite_type"] = results.apply(lambda x: x[1])
    df_perov = df[df["is_perovskite"]].copy()
    print(f"   ✔ Setelah filter formula perovskite : {len(df_perov)}")

    df_perov["crystal_sys_label"] = df_perov[COL_CRYSTAL_SYS].map(CRYSTAL_SYS_MAP)
    df_filtered = df_perov[df_perov[COL_CRYSTAL_SYS].isin(TARGET_CRYSTAL_SYS)].copy()
    print(f"   ✔ Setelah filter crystalSys         : {len(df_filtered)}")
    return df_filtered

df_train_meta = filter_perovskite_df(TRAIN_CSV, label="TRAIN")
df_val_meta   = filter_perovskite_df(VAL_CSV,   label="VAL")




📂 [TRAIN] Dimuat: 553752 baris, 13 kolom
   ✔ Setelah filter formula perovskite : 11568
   ✔ Setelah filter crystalSys         : 8232

📂 [VAL] Dimuat: 138438 baris, 13 kolom
   ✔ Setelah filter formula perovskite : 2892
   ✔ Setelah filter crystalSys         : 2058


Menyimpan Output

In [5]:
df_train_meta.to_csv("df_train_meta_filtered.csv", index=False)
df_val_meta.to_csv("df_val_meta_filtered.csv", index=False)

config_data = {
    "TRAIN_BACK_DIR": TRAIN_BACK_DIR,
    "VAL_BACK_DIR": VAL_BACK_DIR,
    "COL_DATA_ID": COL_DATA_ID,
    "COL_CRYSTAL_SYS": COL_CRYSTAL_SYS,
    "TARGET_CRYSTAL_SYS": TARGET_CRYSTAL_SYS,
    "CRYSTAL_SYS_MAP": CRYSTAL_SYS_MAP,
    "N_POINTS": N_POINTS,
    "EPOCHS": EPOCHS,
    "BATCH_SIZE": BATCH_SIZE,
    "RANDOM_SEED": RANDOM_SEED,
    "PALETTE": PALETTE
}
with open("config.json", "w") as f:
    json.dump(config_data, f)
print("Filter selesai, hasil dan konfigurasi disimpan.")

Filter selesai, hasil dan konfigurasi disimpan.


Validasi Unsur(Kode tambahan)

In [ ]:
ALL_KNOWN = A_SITE | B_SITE | X_SITE

def cek_unknown(csv_path):
    df = pd.read_csv(csv_path)
    semua_unknown = set()
    for formula in df["formula"].dropna():
        elems = set(re.findall(r"[A-Z][a-z]?", str(formula)))
        semua_unknown |= elems - ALL_KNOWN
    print("Elemen tidak dikenali:", semua_unknown)

cek_unknown(TRAIN_CSV)
cek_unknown(VAL_CSV)



Elemen tidak dikenali: {'D', 'Be', 'Kr', 'As', 'C', 'Pu', 'Cm', 'Tc', 'Po', 'Pa', 'Sb', 'Xe', 'Ne', 'H', 'Au', 'Hg', 'Am', 'Np', 'B', 'U'}
Elemen tidak dikenali: {'D', 'Be', 'Kr', 'As', 'C', 'Pu', 'Cm', 'Tc', 'Po', 'Pa', 'Sb', 'Xe', 'Ne', 'H', 'Au', 'Hg', 'Am', 'Np', 'B', 'U'}
